In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
import time 
from functools import partial
from jax import flatten_util
import openfermion
import openfermionpyscf

from VMC_tool import hi, edges,ha,SingleStateAnsatz,create_machine,compute_local_energies,\
    compute_qgt,forces_expect_hermitian,E_fcis

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: To build H|ψ⟩ use nk.vqs.apply_operator(H, vstate_ψ).

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [2]:
from openfermionpyscf import run_pyscf
from openfermion.chem import MolecularData
from openfermion import FermionOperator
from openfermion.transforms import get_fermion_operator   # 关键！
import netket as nk
from netket.operator._fermion2nd import FermionOperator2nd

# 1. 分子
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]

# 2. OpenFermion MolecularData
molecule = MolecularData(
    geometry=geometry,
    basis="sto-3g",
    charge=0,
    multiplicity=1
)
molecule = run_pyscf(molecule, run_scf=True, run_fci=False)

# 3. 拿到 InteractionOperator（含1e、2e积分）
interaction_op = molecule.get_molecular_hamiltonian()

# 4. 官方：InteractionOperator → FermionOperator（二次量子化）
fermion_op = get_fermion_operator(interaction_op)   # ✅ 正规API

# 5. 构造 NetKet 希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=molecule.n_orbitals,
    s=1/2,
    n_fermions=molecule.n_electrons
)

# 6. 导入 NetKet 的二次量子化哈密顿量
ha = FermionOperator2nd.from_openfermion(
    hilbert=hi,
    of_fermion_operator=fermion_op
)


In [3]:
import jax
import jax.numpy as jnp
from functools import partial

# ==============================================
# 1. 生成随机初始态
# ==============================================
def generate_random_initial_states(hi, n_chains: int, seed: int = 42):
    key = jax.random.PRNGKey(seed)
    keys = jax.random.split(key, n_chains)
    return jax.vmap(lambda k: hi.random_state(k))(keys)

# ==============================================
# 2. 候选状态生成
# ==============================================
def make_get_all_next_states(edges):
    @jax.jit
    def get_all_next_states_jit(S: jnp.ndarray):
        next_states = []
        valid_masks = []
        for (i, j) in edges:
            occ_i = S[..., i]
            occ_j = S[..., j]
            valid = (occ_i != occ_j)
            new_state = S.at[..., i].set(occ_j).at[..., j].set(occ_i)
            next_states.append(new_state)
            valid_masks.append(valid)
        return jnp.stack(next_states), jnp.stack(valid_masks)
    return get_all_next_states_jit

# ==============================================
# 3. Metropolis 单步跃迁
# ==============================================
def make_metropolis_hastings_step(edges, machine):
    get_all_next = make_get_all_next_states(edges)
    
    @jax.jit
    def mh_step(params, state: jnp.ndarray, key: jax.Array):
        candidates, valid_mask = get_all_next(state[None, :])
        candidates = candidates[:, 0]
        valid_mask = valid_mask[:, 0]
        
        key, subk = jax.random.split(key)
        idx = jax.random.choice(subk, len(edges))
        cand = candidates[idx]
        is_valid = valid_mask[idx]
        
        log_curr = machine(params, state)
        log_cand = machine(params, cand)
        log_acc = 2 * jnp.real(log_cand - log_curr)
        
        key, subk = jax.random.split(key)
        accept = is_valid & (log_acc > jnp.log(jax.random.uniform(subk)))
        new_state = jnp.where(accept, cand, state)
        return new_state, key
    
    return mh_step

# ==============================================
# 🔥 最终版：带 sampler_state + 随机数管理 + 对齐 NetKet
# ==============================================
@partial(jax.jit, static_argnums=(0,1,3,4,6))
def mcmc_sampler_multichain(
    n_samples_per_chain: int,
    n_warmup: int,             # 单位：sweep
    sampler_state: tuple,      # ✅ NetKet 风格状态：(current_states, chain_keys)
    edges: tuple,
    machine: callable,
    params: dict,
    sweep_size: int = 32       # ✅ 保留 sweep_size
):
    # 解开 sampler_state（和 NetKet 完全一致）
    current_states, current_keys = sampler_state
    n_chains = current_states.shape[0]
    mh_step = make_metropolis_hastings_step(edges, machine)

    # -------------------------
    # 一次 sweep = 连续跳 sweep_size 次
    # -------------------------
    def single_sweep(carry, _):
        states, keys = carry
        # 多链并行 VMAP
        (new_s, new_k), _ = jax.lax.scan(
            lambda c, _: (jax.vmap(mh_step, in_axes=(None, 0, 0))(params, c[0], c[1]), None),
            (states, keys),
            length=sweep_size
        )
        return (new_s, new_k), new_s

    # -------------------------
    # 1) Warmup（仅更新状态，不保存样本）
    # -------------------------
    if n_warmup > 0:
        (current_states, current_keys), _ = jax.lax.scan(
            single_sweep, (current_states, current_keys), length=n_warmup
        )

    # -------------------------
    # 2) 正式采样（保存样本 + 更新最终状态）
    # -------------------------
    (final_states, final_keys), samples = jax.lax.scan(
        single_sweep, (current_states, current_keys), length=n_samples_per_chain
    )

    # 打包新的 sampler_state（返回给下一次迭代）
    new_sampler_state = (final_states, final_keys)
    
    # 展平样本：[n_samples, n_chains, n_sites] → [n_samples*n_chains, n_sites]
    samples_flat = samples.reshape(-1, current_states.shape[-1])
    return samples_flat, new_sampler_state

In [4]:
# 初始化链状态 + 随机数状态（构成 sampler_state）
def init_sampler_state(hi, n_chains, seed=42):
    init_states = generate_random_initial_states(hi, n_chains, seed)
    key = jax.random.PRNGKey(seed)
    chain_keys = jax.random.split(key, n_chains)  # 每条链独立随机数
    return (init_states, chain_keys)

In [5]:
# ======================
# 超参数
# ======================
N_CHAINS = 16
N_WARMUP = 32
N_SAMPLES_PER_CHAIN = 100
SWEEP_SIZE = 32

# ======================
# 初始化 ONCE
# ======================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

sampler_state = init_sampler_state(hi, N_CHAINS, seed=42)
samples, sampler_state = mcmc_sampler_multichain(
    n_samples_per_chain=N_SAMPLES_PER_CHAIN,
    n_warmup=N_WARMUP,
    sampler_state=sampler_state,  # ✅ 状态传递
    edges=((0,1),(2,3)),
    machine=machine,
    params=params,
    sweep_size=SWEEP_SIZE
)
samples.shape

(1600, 4)

In [6]:
@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    最终正确版：无Tracer泄漏 + 维度正确 + 纯JAX
    """
    # 1. 局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    O_mean, O_std = statistics(O_loc)
    O_centered = O_loc - O_mean

    # -------------------------------------------------------------------------
    # 正确方式：对每个样本单独求梯度 (vmap + grad)
    # 这是唯一不会丢维度、不会泄漏、不会报错的写法
    # -------------------------------------------------------------------------
    def log_psi_single(p, s):
        return machine(p, s)

    grad_log_psi = jax.vmap(jax.grad(log_psi_single, holomorphic=True), in_axes=(None, 0))(params, sigma)

    # -------------------------------------------------------------------------
    # 加权平均
    # -------------------------------------------------------------------------
    def weight_and_mean(grad_component):
        weights = O_centered.reshape((-1,) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)

    grad = jax.tree_util.tree_map(weight_and_mean, grad_log_psi)

    return O_mean, O_std, grad

def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])

In [7]:
O_mean, O_std, grad = forces_expect_hermitian(machine,params,hi.all_states())


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/operator/_fermion2nd/jax.py:557: UserWarning: 
Consider using `netket.experimental.operator.ParticleNumberAndSpinConservingFermioperator2nd` to reduce the number of connected elements and
considerably reduce the computational cost.
You can convert this operator by calling `netket.experimental.operator.ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd`.

  super()._setup(self)


In [8]:
import jax
import jax.numpy as jnp
import optax
import time
from functools import partial
import flax.nnx as nnx

# ===================== 6. 初始化（适配多链） =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(params)

# 推荐参数（和 NetKet 一样快、一样准）
N_CHAINS = 100
N_SAMPLES_PER_CHAIN = 20
N_WARMUP = 10
SWEEP_SIZE = 32
N_ITER = 300

# ===================== ✅ 关键：初始化 sampler_state（只初始化一次！）=====================
def init_sampler_state(hi, n_chains, seed=42):
    init_states = generate_random_initial_states(hi, n_chains, seed)
    key = jax.random.PRNGKey(seed)
    chain_keys = jax.random.split(key, n_chains)
    return (init_states, chain_keys)

# 初始化一次，后面永远复用、更新
sampler_state = init_sampler_state(hi, N_CHAINS, seed=21)

# ===================== 7. 训练循环（✅ 完全修复版）=====================
print("\n" + "="*60)
print("开始多链 VMC 训练 (自然梯度下降法)")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}
start_time = time.time()

for step in range(N_ITER):
    # ======================================================================
    # ✅ 1. 采样：复用 sampler_state，不每次重新生成初始态！（核心修复）
    # ======================================================================
    samples, sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,  # 状态传递
        edges=((0,1),(2,3)),
        machine=machine,
        params=params,
        sweep_size=SWEEP_SIZE
    )

    # ======================================================================
    # 2. 能量 & 自然梯度（不变）
    # ======================================================================
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_map(lambda x: x * 2, grad)

    qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.001)
    grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
    natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)

    # ======================================================================
    # 3. 参数更新
    # ======================================================================
    updates, opt_state = optimizer.update(natural_grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    # ======================================================================
    # 4. 日志
    # ======================================================================
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

end_time = time.time()
print(f"\n训练耗时：{end_time - start_time:.2f} 秒")

# ======================================================================
# 最终能量评估
# ======================================================================
final_samples, _ = mcmc_sampler_multichain(
    n_samples_per_chain=N_SAMPLES_PER_CHAIN * 2,
    n_warmup=5,
    sampler_state=sampler_state,
    edges=((0,1),(2,3)),
    machine=machine,
    params=params,
    sweep_size=SWEEP_SIZE
)
final_energy, final_std, _ = forces_expect_hermitian(machine, params, final_samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])

print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)


开始多链 VMC 训练 (自然梯度下降法)


UnexpectedTracerError: Encountered an unexpected tracer. A function transformed by JAX had a side effect, allowing for a reference to an intermediate value with type float64[1] wrapped in a DynamicJaxprTracer to escape the scope of the transformation.
JAX transformations require that functions explicitly return their outputs, and disallow saving intermediate values to global state.
The function being traced when the value leaked was compute_local_energies at /Users/yangjianfei/mac_vscode/神经网络量子态/5 月/0510/JAX 自然梯度 VMC/基态计算/VMC_tool.py:113 traced for jit.
------------------------------
The leaked intermediate value was created on line /Users/yangjianfei/mac_vscode/神经网络量子态/5 月/0510/JAX 自然梯度 VMC/基态计算/VMC_tool.py:120:17 (compute_local_energies). 
------------------------------
When the value was created, the final 5 stack frames (most recent last) excluding JAX-internal frames were:
------------------------------
/Users/yangjianfei/mac_vscode/神经网络量子态/5 月/0510/JAX 自然梯度 VMC/基态计算/VMC_tool.py:1